# Giga Meter — Baseline & ISP Contract Review

Connectivity baseline and ISP performance analysis, parameterised by **education
level** and **year**. Loads the dataset prepared by `gigameter_downloadcleandata.ipynb` — the
cleaning decisions (server, latency cutoff, school-hours window) are inherited,
not repeated.

**Questions answered**
- **Q1** — when were the most schools actively reporting, per year?
- **Q2** — the connectivity baseline per school (+ Q2b year-over-year change, Q2c bandwidth by area)
- **Q3** — ISP performance against an agreed threshold (+ Q3b year-over-year per ISP, Q3c differences between ISPs)
- **Annex** — measurement-validity deep dive and roaming/foreign-carrier flags

Set the scope in the loader cell below (`EDUCATION_LEVEL`, `YEARS`, thresholds).

---
## Part 0 — Load the clean dataset

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================

import os
import sys
import json
from pathlib import Path
from datetime import date, timedelta

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import pytz

from IPython.display import display

# Optional connectivity libs (only needed when USE_CACHED_DATA = False)
try:
    import delta_sharing
    DELTA_SHARING_AVAILABLE = True
except ImportError:
    DELTA_SHARING_AVAILABLE = False
    print("⚠️ delta_sharing not available - will use cached data only")

try:
    import trino
    from trino.dbapi import connect
    TRINO_AVAILABLE = True
except ImportError:
    TRINO_AVAILABLE = False
    print("⚠️ trino not available - will use cached data only")

# -----------------------------------------------------------------------------
# Data-loading helpers (bundled in ./helpers)
#   load_master            - school master via Delta Sharing / Trino, CSV-cached
#   load_measurements      - country measurements via Trino, parquet-cached + incremental
#   format_measurements    - query builder + light post-processing for the
#                            consolidated table default.all_gigameter_measurement_data
#   get_trino_cursor/engine - PRD Trino over kubectl port-forward (auto-started)
# -----------------------------------------------------------------------------
set_up_dir = Path.cwd() / "helpers"
if str(set_up_dir) not in sys.path:
    sys.path.insert(0, str(set_up_dir))

try:
    import format_measurements
    from load_master import load_master, load_master_trino
    from load_measurements import (
        load_measurements,
        load_registration,
        get_trino_cursor,
        get_trino_engine,
    )
    HELPERS_AVAILABLE = True
except ImportError as e:
    HELPERS_AVAILABLE = False
    print(f"⚠️ data-loading helpers not importable from {set_up_dir}: {e}")

# -----------------------------------------------------------------------------
# Analysis helpers + Giga chart style (moved out of the notebook)
#   eda_helpers      - education inference, ISP canonicalisation, IQB-Edu engine,
#                      legacy service-tier scaffolding
#   giga_chart_style - fonts + palette + rcParams (applied on import)
# -----------------------------------------------------------------------------
from eda_helpers import (resolve_country, infer_edlevel_from_name, clean_isp, build_isp_canon,
                         IQB_CONFIG, IQB_USE_CASES, IQB_PERCENTILES, IQB_BENCHMARK,
                         MIN_MEASUREMENTS_FOR_IQB, calculate_iqb_score,
                         _config_for_use_case,
                         classify_service_level, tier_order,
                         TIER_THRESHOLD_1, TIER_THRESHOLD_2, TIER_THRESHOLD_3)
from giga_chart_style import (GIGA_PRIMARY, GIGA_GREY, GIGA_BLUE, GIGA_GOOD,
                              GIGA_MODERATE, GIGA_BAD, GIGA_TIER_RAMP, GIGA_CYCLE,
                              GIGA_SUPTITLE)

print("\u2713 Imports complete \u00b7 Giga chart style applied (Open Sans / Manrope, Giga palette)")

In [ ]:
# =============================================================================
# LOAD THE CLEAN DATASET (produced by gigameter_downloadcleandata.ipynb)
# =============================================================================
# Restores the analysis frames and every parameter chosen during cleaning, so
# this notebook inherits those decisions instead of repeating them.
import json as _json, sys
from pathlib import Path

COUNTRY_NAME = "South Africa"      # <- set the country whose clean dataset to load
CACHE_DIR = f"./cache/{COUNTRY_NAME}"   # cleaned data + caches land here (gitignored)

_slug = COUNTRY_NAME.lower().replace(' ', '')
PARAMS = _json.loads((Path(CACHE_DIR) / f"{_slug}_clean_params.json").read_text())

COUNTRY_ISO3 = PARAMS['country']['iso3']; COUNTRY_ISO2 = PARAMS['country']['iso2']
TIMEZONE = PARAMS['country']['timezone']
ADMIN1_FILTER = PARAMS['filters']['admin1_filter']
MEASUREMENT_SOURCE = PARAMS['filters']['measurement_source']
MAIN_SERVERS = PARAMS['filters'].get('main_servers')
LATENCY_OUTLIER_THRESHOLD = PARAMS['filters']['latency_outlier_threshold_ms']
SCHOOL_HOURS_START = PARAMS['filters']['school_hours_start']
SCHOOL_HOURS_END = PARAMS['filters']['school_hours_end']
MIN_WEEKDAYS_MEASURED = PARAMS['scope_defaults']['min_weekdays_measured']

# ── Analysis scope (override freely — these are the clean-run defaults) ───────
EDUCATION_LEVEL = PARAMS['scope_defaults']['education_level']   # None = all levels
YEARS = PARAMS['scope_defaults']['years']                       # None = auto-detect
MIN_DAYS_MONTH = PARAMS['scope_defaults']['min_days_month']
THR = PARAMS['scope_defaults']['thr_download_mbps']
THR_UL = PARAMS['scope_defaults']['thr_upload_mbps']
THR_LAT = PARAMS['scope_defaults']['thr_latency_ms']
EXPORT_RESULTS = False
OUTPUT_DIR = "./output"

m = pd.read_parquet(Path(CACHE_DIR) / f"{_slug}_clean.parquet")
m_original = pd.read_parquet(Path(CACHE_DIR) / f"{_slug}_clean_unfiltered.parquet")
master = pd.read_csv(Path(CACHE_DIR) / f"{COUNTRY_ISO3}_master_datapull.csv", low_memory=False)
r = pd.read_parquet(Path(CACHE_DIR) / f"{_slug}_registered.parquet")

_wd0 = m['is_weekday'].astype(bool) if 'is_weekday' in m.columns else (m['timestamplocal'].dt.weekday < 5)
m_school = m[(m['measurement_time_window'] == 'school_hours') & _wd0].copy()

print(f"\u2713 {COUNTRY_NAME} ({COUNTRY_ISO3}) clean dataset — prepared {PARAMS['generated_at']}")
print(f"  {len(m):,} measurements ({len(m_school):,} school-hours weekday) from "
      f"{m['school_id_giga'].nunique():,} schools, {PARAMS['window']['first_date']} to {PARAMS['window']['last_date']}")
print(f"  servers={MAIN_SERVERS} · latency cutoff={LATENCY_OUTLIER_THRESHOLD:.0f} ms · "
      f"school hours {SCHOOL_HOURS_START}-{SCHOOL_HOURS_END}")
print(f"  scope: education level={EDUCATION_LEVEL or 'ALL'} · years={YEARS or 'auto'} · "
      f"thresholds {THR}/{THR_UL} Mbps, {THR_LAT} ms")

---
## Q1 — Reporting coverage

In [ ]:
# =============================================================================
# SCHOOLS MEASURING PER MONTH — stacked by education level
# =============================================================================
_mm = m.dropna(subset=['school_id_giga']).copy()
_mm['month'] = _mm['date'].dt.to_period('M').dt.to_timestamp()
_mm['edu'] = _mm['education_level'].fillna('Unknown') if 'education_level' in _mm.columns else 'Unknown'
_tbl = _mm.groupby(['month', 'edu'])['school_id_giga'].nunique().unstack(fill_value=0)
_order = [c for c in ['Pre-Primary', 'Primary', 'Secondary'] if c in _tbl.columns] + \
         sorted(c for c in _tbl.columns if c not in ('Pre-Primary', 'Primary', 'Secondary'))
_tbl = _tbl[_order]

_palette = [GIGA_PRIMARY[300], GIGA_PRIMARY[600], GIGA_PRIMARY[900],
            GIGA_GREY[400], GIGA_GREY[600], GIGA_MODERATE, GIGA_GOOD] + GIGA_CYCLE
fig, ax = plt.subplots(figsize=(13, 5))
_tbl.plot(kind='bar', stacked=True, ax=ax, width=0.85, color=_palette[:len(_tbl.columns)])
ax.set_xticklabels([t.strftime('%Y-%m') for t in _tbl.index], rotation=60, ha='right', fontsize=8)
ax.set_ylabel('schools measuring')
ax.set_title(f'Schools measuring per month by education level — {COUNTRY_NAME}')
ax.legend(title=None, fontsize=9, ncol=2)
plt.tight_layout(); plt.show()

# Richest month — most schools AND most measurement days per school.
# school-days = sum over schools of distinct days measured (breadth x depth).
_mm['d'] = _mm['date'].dt.date
_msum = (_mm.drop_duplicates(['school_id_giga', 'd']).groupby('month')
            .agg(schools=('school_id_giga', 'nunique'), school_days=('school_id_giga', 'size')))
_msum['days_per_school'] = _msum['school_days'] / _msum['schools']
_b_sch, _b_vol, _b_dep = _msum['schools'].idxmax(), _msum['school_days'].idxmax(), _msum['days_per_school'].idxmax()
print(f"Most schools measuring:        {_b_sch:%Y-%m}  ({_msum.loc[_b_sch,'schools']:.0f} schools, "
      f"{_msum.loc[_b_sch,'days_per_school']:.1f} days/school)")
print(f"Richest month (school-days):   {_b_vol:%Y-%m}  ({_msum.loc[_b_vol,'school_days']:,.0f} school-days = "
      f"{_msum.loc[_b_vol,'schools']:.0f} schools x {_msum.loc[_b_vol,'days_per_school']:.1f} days each)")
print(f"Deepest per-school coverage:   {_b_dep:%Y-%m}  ({_msum.loc[_b_dep,'days_per_school']:.1f} days/school "
      f"across {_msum.loc[_b_dep,'schools']:.0f} schools)")

# Time series: days/school/month, each point annotated with n = # schools
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(_msum.index, _msum['days_per_school'], marker='o', ms=5, lw=1.8, color=GIGA_BLUE)
for _mo, _r in _msum.iterrows():
    ax.annotate(f"{_r['schools']:.0f}", (_mo, _r['days_per_school']),
                textcoords='offset points', xytext=(0, 7), ha='center',
                fontsize=7, color=GIGA_GREY[700])
ax.set_ylabel('days / school / month')
ax.set_ylim(0, _msum['days_per_school'].max() * 1.2)
ax.set_title(f'Measurement days per school per month — {COUNTRY_NAME} (labels = schools measuring)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

# Measurement type per day (notes field: daily / startup / manual / first ...)
_nt = m.dropna(subset=['notes']).copy()
_nt['notes_clean'] = _nt['notes'].replace('', 'unlabelled')
_dtbl = _nt.groupby([_nt['date'].dt.date, 'notes_clean']).size().unstack(fill_value=0)
_dtbl = _dtbl[_dtbl.sum().sort_values(ascending=False).index]   # biggest type at the bottom
fig, ax = plt.subplots(figsize=(13, 4))
_x = np.arange(len(_dtbl)); _bottom = np.zeros(len(_dtbl))
for _col, _c in zip(_dtbl.columns, _palette):
    ax.bar(_x, _dtbl[_col].values, bottom=_bottom, width=1.0, label=_col, color=_c, linewidth=0)
    _bottom += _dtbl[_col].values
_ticks = [i for i, d in enumerate(_dtbl.index) if d.day == 1][::2]
ax.set_xticks(_ticks)
ax.set_xticklabels([_dtbl.index[i].strftime('%Y-%m') for i in _ticks], rotation=60, ha='right', fontsize=8)
ax.set_ylabel('measurements / day')
ax.set_title(f'Measurements per day by type (notes) — {COUNTRY_NAME}')
ax.legend(fontsize=9, ncol=2)
plt.tight_layout(); plt.show()

# Average measurements/day by type — past year vs past month
_ref = m['date'].max()
for _label, _days in (('past year', 365), ('past month', 30)):
    _win = _nt[_nt['date'] >= _ref - pd.Timedelta(days=_days)]
    _ndays = _win['date'].dt.date.nunique()
    if _ndays == 0:
        print(f"\n({_label}: no labelled measurements)"); continue
    _avg = (_win.groupby('notes_clean').size() / _ndays).sort_values(ascending=False)
    print(f"\nAvg measurements/day by type — {_label} ({_ndays} active days):")
    for _t, _v in _avg.items():
        print(f"  {_t:12s} {_v:7.1f}/day  ({100 * _v / _avg.sum():.0f}%)")

In [ ]:
# =============================================================================
# Q1 — PEAK REPORTING PERIOD (per year, for the configured education level)
# =============================================================================
# Builds the analysable-month flags used by every downstream cell, derives the
# year list from the data, and reports the richest month in each year.
mv = m.copy()
mv['month'] = mv['date'].dt.to_period('M').astype(str)
_smd = (mv.assign(_d=mv['date'].dt.date).groupby(['school_id_giga', 'month'])['_d']
          .nunique().rename('month_test_days').reset_index())
mv = mv.merge(_smd, on=['school_id_giga', 'month'])
mv['month_qualified'] = mv['month_test_days'] >= MIN_DAYS_MONTH

# Scope roster (education level from the ANALYSIS SCOPE cell; None = all schools)
scope_master = (master[master['education_level'] == EDUCATION_LEVEL].copy()
                if EDUCATION_LEVEL else master.copy())
scope_ids = set(scope_master['school_id_giga'])
mv['in_scope'] = mv['school_id_giga'].isin(scope_ids)
_scope_lbl = EDUCATION_LEVEL or 'all levels'

# Shared school-hours weekday frame used by Q2c/Q3/Q3b/Q3c
_wd = mv['is_weekday'].astype(bool) if 'is_weekday' in mv.columns else (mv['timestamplocal'].dt.weekday < 5)
mv_school = mv[(mv['measurement_time_window'] == 'school_hours') & _wd].copy()
mv_school['week'] = mv_school['timestamplocal'].dt.tz_localize(None).dt.to_period('W-SUN').dt.start_time

# Years present in the data (unless pinned in the scope cell)
YEARS_USED = YEARS or sorted(mv['month'].str[:4].unique())
BASELINE_YEAR = YEARS_USED[0]
COMPARE_YEAR = YEARS_USED[-1] if len(YEARS_USED) > 1 else None
print(f"Years in data: {', '.join(YEARS_USED)}  ->  baseline {BASELINE_YEAR}"
      f"{f', compared against {COMPARE_YEAR}' if COMPARE_YEAR else ''}")
print(f"Roster in scope ({_scope_lbl}): {len(scope_master):,} schools\n")

def _cov(df):
    g = df.groupby('month').agg(schools=('school_id_giga', 'nunique'),
                                n_tests=('school_id_giga', 'size'))
    q = (df[df['month_qualified']].groupby('month')['school_id_giga'].nunique()
         .rename('analysable'))
    return g.join(q).fillna(0)

cov = _cov(mv).join(_cov(mv[mv['in_scope']]).add_prefix('scope_')).fillna(0).astype(int)

RICH_BY_YEAR = {}
for _y in YEARS_USED:
    _cy = cov.loc[cov.index.str.startswith(_y)]
    if _cy.empty or _cy['scope_analysable'].max() == 0:
        print(f"{_y}: no analysable months in scope"); continue
    _r = _cy['scope_analysable'].idxmax()
    RICH_BY_YEAR[_y] = _r
    print(f"{_y}: richest month = {_r} — {_cy.loc[_r, 'scope_analysable']} analysable "
          f"{_scope_lbl} schools of {len(scope_master)} in the roster "
          f"({_cy.loc[_r, 'scope_schools']} reported at all); "
          f"all schools that month: {_cy.loc[_r, 'analysable']} analysable")
RICH = RICH_BY_YEAR.get(BASELINE_YEAR)
display(cov[cov.index.str[:4].isin(YEARS_USED)])

---
## Q2 — Connectivity baseline

In [ ]:
# =============================================================================
# Q2 — BASELINE PER SCHOOL (modular by education level and year)
# =============================================================================

# thresholds come from the ANALYSIS SCOPE cell

def get_school_baseline(
    mv, 
    master, 
    education_level=None,        # str: 'Secondary', 'Primary', etc., or None for all
    year=None,                   # str or None. Example: '2025', '2026', or None for all years
    cache_dir=CACHE_DIR,         # for saving
    superset_filename='zaf_school_baseline.csv' # for saving
):
    """
    Compute baseline stats for schools by education level and year.
    Returns: (base, gap, scope_master, sch_sel)
    """
    # Filtering by education level in master
    if education_level is not None:
        edumaster = master[master['education_level'] == education_level].copy()
    else:
        edumaster = master.copy()
    school_ids = set(edumaster['school_id_giga'])

    # Filter mv for weekday school-hours, and schools in education_level (if specified)
    _wd = mv['is_weekday'].astype(bool) if 'is_weekday' in mv.columns else (mv['timestamplocal'].dt.weekday < 5)
    mv_school = mv[(mv['measurement_time_window'] == 'school_hours') & _wd].copy()
    mv_school['week'] = mv_school['timestamplocal'].dt.tz_localize(None).dt.to_period('W-SUN').dt.start_time
    if education_level is not None:
        mv_school = mv_school[mv_school['school_id_giga'].isin(school_ids)]

    # Filter for selected year (if provided)
    if year is not None:
        year_sel = mv_school['month'].str.startswith(str(year))
    else:
        year_sel = pd.Series([True] * len(mv_school), index=mv_school.index)
    sch_sel = mv_school[year_sel & mv_school['month_qualified']]

    def _mode_isp(s):
        s = s.dropna()
        return s.mode().iloc[0] if len(s) else np.nan

    # Baseline aggregation per school
    base = (sch_sel.groupby('school_id_giga')
            .agg(school_name=('school_name', 'first'),
                 n_tests=('download_speed', 'size'),
                 test_days=('date', lambda s: s.dt.date.nunique()),
                 dl_p5=('download_speed', lambda s: s.quantile(0.05)),
                 dl_p50=('download_speed', 'median'),
                 dl_p95=('download_speed', lambda s: s.quantile(0.95)),
                 ul_p50=('upload_speed', 'median'),
                 lat_p50=('latency', 'median'),
                 loss_p50=('loss_rate', 'median'),
                 primary_isp=('isp_mapped', _mode_isp),
                 pct_tests_below_thr=('download_speed', lambda s: 100 * (s < THR).mean())))
    _wp = (sch_sel.groupby(['school_id_giga', 'week'])['download_speed']
           .quantile(0.95).ge(THR).groupby(level=0).mean() * 100)
    base = base.join(_wp.rename('pct_weeks_p95_ge_thr'))
    base = base.merge(master.set_index('school_id_giga')[['admin1']],
                      left_index=True, right_index=True, how='left')

    # Gap: schools present in master but no analysable data in period
    gap = edumaster.loc[~edumaster['school_id_giga'].isin(base.index),
                        ['school_id_giga', 'school_name', 'admin1', 'connectivity']]

    return base, gap, edumaster, sch_sel

def print_baseline_report(base, gap, edumaster, year=None, education_level=None):
    elvl = f" ({education_level})" if education_level else ""
    print(f"Schools with a {year} baseline{elvl} (analysable months only): {len(base)} of {len(edumaster)}")
    print(f"National school baseline (median of school medians): "
          f"download {base['dl_p50'].median():.1f} Mbps · upload {base['ul_p50'].median():.1f} Mbps · "
          f"latency {base['lat_p50'].median():.0f} ms · packet loss {base['loss_p50'].median():.2%} "
          f"(retrans-ratio proxy, {base['loss_p50'].notna().sum()} of {len(base)} schools with data)")
    print(f"Schools with median download < {THR} Mbps: {(base['dl_p50'] < THR).sum()} of {len(base)} "
          f"({100 * (base['dl_p50'] < THR).mean():.0f}%)")
    print(f"Schools with median upload < {THR_UL} Mbps: {(base['ul_p50'] < THR_UL).sum()} of {len(base)} "
          f"({100 * (base['ul_p50'] < THR_UL).mean():.0f}%)")

def plot_baseline_trends(base, mmed=None, year=None, education_level=None):
    # Sparkline function
    

    def _spark(vals, w=90, h=18, color=GIGA_PRIMARY[600]):
        _v = [x for x in vals if pd.notna(x)]
        if len(_v) < 2:
            return ''
        _lo, _hi = min(_v), max(_v)
        _rng = (_hi - _lo) or 1
        _pts = ' '.join(f"{_k * w / (len(_v) - 1):.1f},{h - (x - _lo) / _rng * (h - 2) - 1:.1f}"
                        for _k, x in enumerate(_v))
        return (f'<svg width="{w}" height="{h}" style="vertical-align:middle">'
                f'<polyline points="{_pts}" fill="none" stroke="{color}" stroke-width="1.4"/></svg>')

    # Monthly medians for sparklines, if provided
    _tbl = base.sort_values('dl_p50').head(15).copy()
    if mmed is not None:
        _tbl.insert(2, 'dl_trend', [_spark(mmed.loc[_s].tolist()) if _s in mmed.index else ''
                                    for _s in _tbl.index])
    _numcols = ['dl_p5', 'dl_p50', 'dl_p95', 'ul_p50', 'lat_p50', 'loss_p50',
                'pct_tests_below_thr', 'pct_weeks_p95_ge_thr']
    _styled = (_tbl.style
               .format({c: '{:.1f}' for c in _numcols if c != 'loss_p50'} | {'loss_p50': '{:.2%}'})
               .background_gradient(cmap='RdYlGn', subset=['dl_p5', 'dl_p50', 'dl_p95', 'ul_p50',
                                                           'pct_weeks_p95_ge_thr'])
               .background_gradient(cmap='RdYlGn_r', subset=['lat_p50', 'loss_p50',
                                                             'pct_tests_below_thr'])
               .set_properties(**{'font-size': '11px'}))
    print("Lowest-download schools (colour: green = better; sparkline = monthly trend):")
    display(_styled)

    # Plot barplots by metric
    # Download
    _b = base.sort_values('dl_p50').reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.bar(_b.index, _b['dl_p50'], width=1.0,
           color=[GIGA_BAD if v < THR else GIGA_GOOD for v in _b['dl_p50']])
    ax.axhline(THR, color=GIGA_GREY[800], ls='--', lw=1)
    _cap = 100
    _over = (_b['dl_p50'] > _cap).sum()
    ax.set_ylim(0, _cap)
    if _over:
        ax.text(0.99, 0.97, f'{_over} schools above {_cap} Mbps (bars clipped)',
                transform=ax.transAxes, ha='right', va='top', fontsize=9, color=GIGA_GREY[700])
    ax.set_xlabel('schools'); ax.set_ylabel(f'median download {year} (Mbps)')
    ax.set_title(f"{year} baseline — median download per school "
                 f"({(_b['dl_p50'] < THR).sum()} below {THR} Mbps){f' — {COUNTRY_NAME}' if 'COUNTRY_NAME' in globals() else ''}")
    plt.tight_layout(); plt.show()

    # Upload
    _bu = base.sort_values('ul_p50').reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(13, 4))
    ax.bar(_bu.index, _bu['ul_p50'], width=1.0,
           color=[GIGA_BAD if v < THR_UL else GIGA_GOOD for v in _bu['ul_p50']])
    ax.axhline(THR_UL, color=GIGA_GREY[800], ls='--', lw=1)
    _cap_u = 40
    _over_u = (_bu['ul_p50'] > _cap_u).sum()
    ax.set_ylim(0, _cap_u)
    if _over_u:
        ax.text(0.99, 0.97, f'{_over_u} schools above {_cap_u} Mbps (bars clipped)',
                transform=ax.transAxes, ha='right', va='top', fontsize=9, color=GIGA_GREY[700])
    ax.set_xlabel('schools'); ax.set_ylabel(f'median upload {year} (Mbps)')
    ax.set_title(f"{year} baseline — median upload per school "
                 f"({(_bu['ul_p50'] < THR_UL).sum()} below {THR_UL} Mbps){f' — {COUNTRY_NAME}' if 'COUNTRY_NAME' in globals() else ''}")
    plt.tight_layout(); plt.show()

    # Latency
    _bl = base.sort_values('lat_p50').reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(13, 4))
    ax.bar(_bl.index, _bl['lat_p50'], width=1.0,
           color=[GIGA_BAD if v > THR_LAT else GIGA_GOOD for v in _bl['lat_p50']])
    ax.axhline(THR_LAT, color=GIGA_GREY[800], ls='--', lw=1)
    _cap_l = 400
    _over_l = (_bl['lat_p50'] > _cap_l).sum()
    ax.set_ylim(0, _cap_l)
    if _over_l:
        ax.text(0.99, 0.97, f'{_over_l} schools below {_cap_l} Ms (bars clipped)',
                transform=ax.transAxes, ha='right', va='top', fontsize=9, color=GIGA_GREY[700])
    ax.set_xlabel('schools'); ax.set_ylabel(f'median latency {year} (Ms)')
    ax.set_title(f"{year} baseline — median latency per school "
                 f"({(_bl['lat_p50'] < THR_LAT).sum()} below {THR_LAT} Ms){f' — {COUNTRY_NAME}' if 'COUNTRY_NAME' in globals() else ''}")
    plt.tight_layout(); plt.show()

def save_baseline(base, cache_dir, filename):
    _outd = Path(cache_dir) / 'superset'
    _outd.mkdir(exist_ok=True)
    base.reset_index().round(2).to_csv(_outd / filename, index=False)
    print(f"\nSaved: {_outd / filename}  ({len(base)} schools)")

def show_gap(gap):
    if len(gap):
        print(f"Schools with NO analysable month ({len(gap)}):")
        display(gap.head(25))

# --- RUN FOR THE CONFIGURED SCOPE (see ANALYSIS SCOPE cell) ---
base, gap, scope_master, sch_sel = get_school_baseline(
    mv, master, education_level=EDUCATION_LEVEL, year=BASELINE_YEAR)
print_baseline_report(base, gap, scope_master, year=BASELINE_YEAR, education_level=EDUCATION_LEVEL)
mmed = (sch_sel.groupby(['school_id_giga', 'month'])['download_speed']
        .median().unstack().sort_index(axis=1))
plot_baseline_trends(base, mmed=mmed, year=BASELINE_YEAR, education_level=EDUCATION_LEVEL)
show_gap(gap)

_lvl_tag = (EDUCATION_LEVEL or 'all').lower().replace(' ', '')
_yr_tag = BASELINE_YEAR or 'allyears'
save_baseline(base, CACHE_DIR, f"{COUNTRY_ISO3.lower()}_school_baseline_{_lvl_tag}_{_yr_tag}.csv")


In [ ]:
# =============================================================================
# Q2b — DOES THE BASELINE CHANGE BETWEEN YEARS? (paired per-school comparison)
# =============================================================================
# Compares any two years for the configured education level, using the same
# baseline rules. PAIRED per school (only schools with a baseline in both years).
# Statistical methods, per metric:
#  - Wilcoxon signed-rank: primary test — nonparametric, robust to speed skew.
#  - Paired t-test: parametric companion (skew makes it less reliable here).
#  - Sign test (exact binomial): weakest assumptions — is improvement 50/50?
#  - Rank-biserial r: effect SIZE (0 = none, ±1 = all schools moved one way).
#  - Holm correction across the metrics tested; verdict uses adjusted p < 0.05.
from scipy import stats as _st

def compare_baseline_years(mv, master, year_a, year_b, education_level=None):
    _a_base, _, _, _ = get_school_baseline(mv, master, education_level=education_level, year=year_a)
    _b_base, _, _, _ = get_school_baseline(mv, master, education_level=education_level, year=year_b)
    _cols = ['dl_p50', 'ul_p50', 'lat_p50', 'loss_p50']
    paired = _a_base[_cols].join(_b_base[_cols], lsuffix=f'_{year_a}', rsuffix=f'_{year_b}', how='inner')
    print(f"Schools with an analysable baseline in BOTH {year_a} and {year_b} "
          f"({education_level or 'all levels'}): {len(paired)} "
          f"({year_a} only: {len(_a_base) - len(paired)}, new in {year_b}: {len(_b_base) - len(paired)})")
    if len(paired) < 8:
        print("Too few paired schools for a meaningful test."); return paired, None

    _METRICS = [('dl_p50', 'download (Mbps)', True), ('ul_p50', 'upload (Mbps)', True),
                ('lat_p50', 'latency (ms)', False), ('loss_p50', 'packet loss (retrans ratio)', False)]
    _res = []
    for _mcol, _lbl, _hg in _METRICS:
        _m0 = paired[f'{_mcol}_{year_a}'].notna() & paired[f'{_mcol}_{year_b}'].notna()
        _x, _y = paired.loc[_m0, f'{_mcol}_{year_a}'], paired.loc[_m0, f'{_mcol}_{year_b}']
        if len(_x) < 8:
            continue
        _delta = _y - _x
        _impr = (_delta > 0) if _hg else (_delta < 0)
        try:
            _w = _st.wilcoxon(_x, _y); _wp = _w.pvalue
            _nz = int((_delta != 0).sum())
            _rb = 1 - 2 * _w.statistic / (_nz * (_nz + 1) / 2) if _nz else float('nan')
        except ValueError:
            _wp, _rb = float('nan'), float('nan')
        _np_, _nn = int((_delta > 0).sum()), int((_delta < 0).sum())
        _res.append(dict(metric=_lbl, n=len(_x), med_a=_x.median(), med_b=_y.median(),
                         med_change=_delta.median(), pct_improved=100 * _impr.mean(),
                         wilcoxon_p=_wp, ttest_p=_st.ttest_rel(_x, _y).pvalue,
                         sign_p=_st.binomtest(_np_, _np_ + _nn, 0.5).pvalue if (_np_ + _nn) else float('nan'),
                         effect_r=_rb, higher_good=_hg))
    _order = sorted(range(len(_res)), key=lambda k: _res[k]['wilcoxon_p'])
    _run = 0.0
    for _rank, _k in enumerate(_order):
        _run = min(1.0, max(_run, (len(_res) - _rank) * _res[_k]['wilcoxon_p']))
        _res[_k]['wilcoxon_p_holm'] = _run
    tbl = pd.DataFrame(_res).set_index('metric')
    tbl['significant'] = tbl['wilcoxon_p_holm'] < 0.05
    tbl = tbl.rename(columns={'med_a': f'med_{year_a}', 'med_b': f'med_{year_b}'})
    display(tbl[[f'med_{year_a}', f'med_{year_b}', 'n', 'med_change', 'pct_improved',
                 'wilcoxon_p', 'wilcoxon_p_holm', 'ttest_p', 'sign_p', 'effect_r',
                 'significant']].round(4))
    _sig = tbl[tbl['significant']]
    if len(_sig):
        _parts = [f"{_l} {'improved' if ((r['med_change'] > 0) == r['higher_good']) else 'worsened'} "
                  f"(Holm p={r['wilcoxon_p_holm']:.4f}, r={r['effect_r']:.2f})"
                  for _l, r in _sig.iterrows()]
        print(f"VERDICT ({year_a} -> {year_b}): significant change in " + "; ".join(_parts) + ".")
    else:
        print(f"VERDICT ({year_a} -> {year_b}): no statistically significant change "
              f"(all Holm-adjusted p >= 0.05).")

    fig, ax = plt.subplots(figsize=(10, 3.8))
    _d = paired[f'dl_p50_{year_b}'] - paired[f'dl_p50_{year_a}']
    ax.hist(_d.clip(-30, 30), bins=40, color=GIGA_PRIMARY[600], alpha=0.85, edgecolor='white')
    ax.axvline(0, color=GIGA_GREY[800], lw=1)
    ax.axvline(_d.median(), color=GIGA_BAD if _d.median() < 0 else GIGA_GOOD, ls='--', lw=1.5,
               label=f'median change {_d.median():+.1f} Mbps')
    ax.set_xlabel(f'per-school change in median download, {year_b} vs {year_a} (Mbps, clipped ±30)')
    ax.set_ylabel('schools'); ax.legend()
    ax.set_title(f"Baseline shift {year_a} -> {year_b} ({education_level or 'all levels'}) — {COUNTRY_NAME}")
    plt.tight_layout(); plt.show()
    return paired, tbl

if COMPARE_YEAR:
    paired, year_stats = compare_baseline_years(mv, master, BASELINE_YEAR, COMPARE_YEAR,
                                                education_level=EDUCATION_LEVEL)
    print(f"\nNOTE: {COMPARE_YEAR} may be a partial year — seasonal composition differs.")
else:
    print("Only one year of data in scope — no year-over-year comparison.")

In [ ]:
# =============================================================================
# Q2c — BANDWIDTH BY AREA: H3 HEX TILES AROUND SCHOOLS
# =============================================================================
# Median available bandwidth per hexagonal tile, so "where is connectivity poor"
# is answerable by AREA rather than by school. School-first: each school's own
# median, then the median across schools in the tile (a tile is not dominated by
# one heavy-testing school).
#
# giga-spatial reuse note: gigaspatial.grid.h3.H3Hexagons offers from_points() /
# to_geodataframe() over this same `h3` library, but it lives in a separate
# py310 venv (qos-venv-py310), not this kernel. We call `h3` + geopandas
# directly — same tiling, no extra dependency. If you later run inside that venv,
# H3Hexagons.from_points(points, resolution).to_geodataframe() replaces the
# geometry block below.
import h3, geopandas as gpd
from shapely.geometry import Polygon

H3_RESOLUTION = 5      # ~10 km edge; 4 ~26 km, 6 ~3.7 km — pick per country size
MIN_SCHOOLS_HEX = 3    # tiles with fewer schools are shown faded (not reliable)
HEX_SCOPE_ALL_LEVELS = True   # True = all schools; False = only EDUCATION_LEVEL

_src = mv_school if HEX_SCOPE_ALL_LEVELS else mv_school[mv_school['in_scope']]
_src = _src[_src['month_qualified']].dropna(subset=['latitude', 'longitude', 'download_speed'])
_sch_geo = _src.groupby('school_id_giga').agg(
    school_name=('school_name', 'first'), lat=('latitude', 'median'), lon=('longitude', 'median'),
    dl=('download_speed', 'median'), ul=('upload_speed', 'median'), lat_ms=('latency', 'median'),
    n_tests=('download_speed', 'size'))
_sch_geo['hex'] = [h3.latlng_to_cell(_la, _lo, H3_RESOLUTION)
                   for _la, _lo in zip(_sch_geo['lat'], _sch_geo['lon'])]

hex_bw = (_sch_geo.groupby('hex')
          .agg(schools=('dl', 'size'), dl_med=('dl', 'median'), dl_min=('dl', 'min'),
               dl_max=('dl', 'max'), ul_med=('ul', 'median'), lat_med=('lat_ms', 'median'))
          .reset_index())
hex_bw['below_thr'] = hex_bw['dl_med'] < THR
hex_bw['geometry'] = [Polygon([(_lng, _lat) for _lat, _lng in h3.cell_to_boundary(_h)])
                      for _h in hex_bw['hex']]
hex_gdf = gpd.GeoDataFrame(hex_bw, geometry='geometry', crs='EPSG:4326')
_rel = hex_gdf[hex_gdf['schools'] >= MIN_SCHOOLS_HEX]
_thin = hex_gdf[hex_gdf['schools'] < MIN_SCHOOLS_HEX]
print(f"H3 resolution {H3_RESOLUTION} (~{h3.average_hexagon_edge_length(H3_RESOLUTION, unit='km'):.1f} km edge): "
      f"{len(hex_gdf)} tiles covering {len(_sch_geo)} schools; "
      f"{len(_rel)} tiles with >= {MIN_SCHOOLS_HEX} schools")
print(f"Tiles below the {THR} Mbps threshold: {int(_rel['below_thr'].sum())} of {len(_rel)} reliable tiles")

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
_vmax = float(np.nanpercentile(_rel['dl_med'], 95)) if len(_rel) else 1
if len(_thin):
    _thin.plot(ax=axes[0], color=GIGA_GREY[200], edgecolor='white', linewidth=0.3)
_rel.plot(column='dl_med', cmap='RdYlGn', vmin=0, vmax=_vmax, legend=True,
          edgecolor='white', linewidth=0.4, ax=axes[0],
          legend_kwds={'label': 'median download (Mbps)', 'shrink': 0.6})
axes[0].set_title(f"Median download by area — H3 res {H3_RESOLUTION} — {COUNTRY_NAME}\n"
                  f"(grey = < {MIN_SCHOOLS_HEX} schools; scale capped at p95)", fontsize=11)
_rel.plot(column='schools', cmap='Blues', legend=True, edgecolor='white', linewidth=0.4,
          ax=axes[1], legend_kwds={'label': 'schools in tile', 'shrink': 0.6})
axes[1].set_title('Schools per tile (coverage/reliability)', fontsize=11)
for _ax in axes:
    _ax.set_xlabel('longitude'); _ax.set_ylabel('latitude')
plt.tight_layout(); plt.show()

print(f"\nLowest-bandwidth areas (tiles with >= {MIN_SCHOOLS_HEX} schools):")
display(_rel.drop(columns='geometry').sort_values('dl_med')
        .head(12).set_index('hex').round(1))

if EXPORT_RESULTS:
    _outd = Path(CACHE_DIR) / 'superset'; _outd.mkdir(exist_ok=True)
    hex_gdf.drop(columns='geometry').to_csv(
        _outd / f"{COUNTRY_ISO3.lower()}_bandwidth_by_hex_res{H3_RESOLUTION}.csv", index=False)
    print(f"exported {COUNTRY_ISO3.lower()}_bandwidth_by_hex_res{H3_RESOLUTION}.csv")

---
## Q3 — ISP performance vs threshold

In [ ]:
# =============================================================================
# ISP SUMMARY (top providers + median performance) -- deep dive in the appendix
# =============================================================================
if "isp_mapped" in m.columns and m["isp_mapped"].notna().any():
    _top = m["isp_mapped"].value_counts().head(8).index
    isp_summary = (m[m["isp_mapped"].isin(_top)]
                   .groupby("isp_mapped")
                   .agg(measurements=("isp_mapped", "size"),
                        schools=("school_id_giga", "nunique"),
                        dl_median=("download_speed", "median"),
                        ul_median=("upload_speed", "median"),
                        latency_median=("latency", "median"),
                        loss_median=("loss_rate", "median"))
                   .sort_values("measurements", ascending=False)
                   .round({"dl_median": 1, "ul_median": 1, "latency_median": 0, "loss_median": 4}))
    print("Top ISPs by measurement volume:")
    display(isp_summary)
else:
    print("(no detected_isp data)")

# NOTE: dl_median above is pooled across ALL measurements (volume-weighted —
# heavy-testing schools dominate). Below: the SCHOOL-level view — each school's
# own median, distribution across the ISP's schools.
_pair = (m.dropna(subset=["isp_mapped"])
         .groupby(["isp_mapped", "school_id_giga"])[["download_speed", "upload_speed", "latency"]]
         .median().rename(columns={"download_speed": "dl", "upload_speed": "ul", "latency": "lat"})
         .reset_index())
_pair = _pair[_pair["isp_mapped"].isin(_top)]
def _school_dist(df, by):
    return (df.groupby(by)
            .agg(schools=("dl", "size"),
                 dl_p5=("dl", lambda s: s.quantile(0.05)), dl_p50=("dl", "median"), dl_p90 = ("dl", lambda s: s.quantile(0.90)), dl_p95=("dl", lambda s: s.quantile(0.95)),
                 ul_p5=("ul", lambda s: s.quantile(0.05)), ul_p50=("ul", "median"), ul_p90 = ("ul", lambda s: s.quantile(0.90)), ul_p95=("ul", lambda s: s.quantile(0.95)),
                 lat_p5=("lat", lambda s: s.quantile(0.05)), lat_p50=("lat", "median"), lat_p90 = ("lat", lambda s: s.quantile(0.90)),lat_p95=("lat", lambda s: s.quantile(0.95))))
_dist = _school_dist(_pair, "isp_mapped").sort_values("dl_p50", ascending=False).round(1)
print("\nPer-school medians — distribution across each ISP's schools (dl/ul Mbps, lat ms):")
display(_dist)

_order = [i for i in _dist.index if _dist.loc[i, "schools"] >= 5]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (_col, _lab) in zip(axes, [("dl", "download (Mbps)"), ("ul", "upload (Mbps)"), ("lat", "latency (ms)")]):
    _bp = ax.boxplot([_pair.loc[_pair["isp_mapped"] == i, _col].dropna() for i in _order],
                     tick_labels=[str(i)[:14] for i in _order], showfliers=False, patch_artist=True)
    for _p in _bp["boxes"]:
        _p.set(facecolor=GIGA_PRIMARY[300], edgecolor=GIGA_PRIMARY[700])
    for _med in _bp["medians"]:
        _med.set(color=GIGA_PRIMARY[800], linewidth=1.8)
    ax.set_ylabel(f"school median {_lab}")
    ax.tick_params(axis="x", rotation=45)
fig.suptitle(f"Per-school medians by ISP (boxes = school distribution) — {COUNTRY_NAME}")
plt.tight_layout(); plt.show()

if "detected_server" in m.columns and m["detected_server"].notna().any():
    print("\nTop M-Lab server locations:")
    display(m["detected_server"].value_counts().head(5).rename("measurements").to_frame())

In [ ]:
# =============================================================================
# Q3 — WEEKLY THRESHOLD COMPLIANCE BY ISP (CC-aligned)
# =============================================================================
# For composite compliance: dl_p95 >= THR, ul_p95 >= 10, lat_p95 < 100

# mv_school (school-hours weekday + week) is built in the Q1 cell
sw = (mv_school[mv_school['month'].str[:4].isin(YEARS_USED) & mv_school['month_qualified']]
      .dropna(subset=['isp_mapped'])
      .groupby(['school_id_giga', 'week', 'isp_mapped'])
      .agg(n_tests=('download_speed', 'size'),
           dl_p50=('download_speed', 'median'),
           ul_p50=('upload_speed', 'median'),
           lat_p50=('latency', 'median'))
      .reset_index())

# Add upload and latency compliance
sw['meets_dl_20'] = sw['dl_p50'] >= 20
sw['meets_ul_10'] = sw['ul_p50'] >= 10
sw['meets_lat_100'] = sw['lat_p50'] < 100

# Composite compliance: all together, e.g., for 20 Mbps
sw['composite_meet'] = sw['meets_dl_20'] & sw['meets_ul_10'] & sw['meets_lat_100']

sw['in_scope'] = sw['school_id_giga'].isin(scope_ids)   # education level from ANALYSIS SCOPE

# Aggregate ISP stats, add composite compliance
isp_rank = (
    sw.groupby('isp_mapped')
      .agg(
          schools=('school_id_giga', 'nunique'),
          school_weeks=('week', 'size'),
          dl_p50_median=('dl_p50', 'median'),
          ul_p50_median=('ul_p50', 'median'),
          lat_p50_median=('lat_p50', 'median'),
          pct_weeks_meet_dl_20=('meets_dl_20', lambda s: 100*s.mean()),
          pct_weeks_meet_ul_10=('meets_ul_10', lambda s: 100 * s.mean()),
          pct_weeks_meet_lat_100=('meets_lat_100', lambda s: 100 * s.mean()),
          pct_weeks_composite=('composite_meet', lambda s: 100 * s.mean()),
      )
)

_chronic = (sw.groupby(['isp_mapped', 'school_id_giga'])['meets_dl_20'].mean().lt(0.5)
            .groupby(level=0).sum())
isp_rank = isp_rank.join(_chronic.rename('schools_chronic_below_20'))
isp_rank = (isp_rank[isp_rank['schools'] >= 5]
            .sort_values('pct_weeks_meet_dl_20', ascending=False).round(1))
print("Per-ISP weekly threshold compliance (school-hours weekday, analysable months):")
display(isp_rank)

# Composite bar chart for download, upload, latency, and composite compliance

fig, axes = plt.subplots(1, 2, figsize=(17, 5))
_r = isp_rank.sort_values('pct_weeks_composite')
from matplotlib.colors import LinearSegmentedColormap as _LSC
_RAG = _LSC.from_list('rag', [GIGA_BAD, GIGA_MODERATE, GIGA_GOOD])

# Composite stacked barh chart (axes[0])
bar_labels = [
    f'% weeks: DL >= {THR} Mbps', 
    '% weeks: UL >= 10 Mbps', 
    '% weeks: Lat < 100ms', 
    '% weeks: All criteria'
]
colors = [
    _RAG(0.5),
    _RAG(0.8),
    _RAG(0.3),
    GIGA_GOOD
]
# barh for each, side-by-side for each ISP

bar_width = 0.18
y_pos = range(len(_r))
axes[0].barh([y + 1.5*bar_width for y in y_pos], _r['pct_weeks_meet_dl_20'],    height=bar_width, label=bar_labels[0], color=colors[0])
axes[0].barh([y + 0.5*bar_width for y in y_pos], _r['pct_weeks_meet_ul_10'],  height=bar_width, label=bar_labels[1], color=colors[1])
axes[0].barh([y - 0.5*bar_width for y in y_pos], _r['pct_weeks_meet_lat_100'],height=bar_width, label=bar_labels[2], color=colors[2])
axes[0].barh([y - 1.5*bar_width for y in y_pos], _r['pct_weeks_composite'],height=bar_width, label=bar_labels[3], color=colors[3])
axes[0].set_yticks(range(len(_r)))
axes[0].set_yticklabels(_r.index)
axes[0].set_xlabel("Percent of school-weeks")
axes[0].set_title(f'Weekly threshold compliance by ISP ({chr(39).join([])}' + ', '.join(YEARS_USED) + ')')
axes[0].legend(loc='lower right', fontsize=9)
axes[0].set_xlim(0, 105)
for _i, idx in enumerate(_r.index):
    for j, val in enumerate([
        _r.loc[idx, 'pct_weeks_meet_dl_20'],
        _r.loc[idx, 'pct_weeks_meet_ul_10'],
        _r.loc[idx, 'pct_weeks_meet_lat_100'],
        _r.loc[idx, 'pct_weeks_composite'],
    ]):
        # Show text labels for composite only (optional: for each bar, if desired)
        if j == 3:
            axes[0].text(val + 1, _i - 1.5*bar_width, f'{val:.0f}%', va='center', fontsize=9, fontweight='bold', color=GIGA_GOOD)

# Line chart (axes[1]) — as before
_top_isps = isp_rank.index[:6]
_mi = (mv_school[mv_school['isp_mapped'].isin(_top_isps)
                 & mv_school['month'].str[:4].isin(YEARS_USED) & mv_school['month_qualified']]
       .groupby(['month', 'isp_mapped'])['download_speed'].median().unstack())
_mi.plot(ax=axes[1], marker='o', ms=3, lw=1.6)
axes[1].axhline(THR, ls='--', lw=1)
axes[1].set_ylabel('median download (Mbps)'); axes[1].set_xlabel('')
axes[1].set_title('Monthly median download by ISP')
axes[1].tick_params(axis='x', rotation=60); axes[1].legend(fontsize=8, ncol=2)
plt.tight_layout(); plt.show()

print(f"NOTE: Starlink schools span an enormous range (p5 ~0.5 Mbps) — 'switch to"
      f"\nStarlink' needs per-site checks. Kacific compliance reflects only its"
      f"\nvalidity-passing tests (76% fail) — its true performance is worse than shown.")

In [ ]:
# =============================================================================
# Q3b — ISP PERFORMANCE BETWEEN YEARS: IS THE CHANGE SIGNIFICANT? (paired)
# =============================================================================
# Paired within ISP: only schools that kept the SAME primary ISP in both years,
# each compared against itself on median download / upload / latency and on
# weekly P95 >= THR compliance. Wilcoxon signed-rank per ISP, Holm-corrected
# across ISPs within each metric. Schools that switched provider are excluded
# from the test but counted below (the churn ledger).
YEAR_A, YEAR_B = BASELINE_YEAR, COMPARE_YEAR
_Q3B_METRICS = [('dl', 'download_speed', 'median download (Mbps)'),
                ('ul', 'upload_speed', 'median upload (Mbps)'),
                ('lat', 'latency', 'median latency (ms)'),
                ('comp', None, f'% school-weeks P95 >= {THR} Mbps')]

if not YEAR_B:
    print("Only one year of data in scope — no ISP year-over-year comparison.")
else:
    _sy = (mv_school[mv_school['month_qualified']
                     & mv_school['month'].str[:4].isin([YEAR_A, YEAR_B])]
           .assign(year=lambda d: d['month'].str[:4]))
    _pisp_y = (_sy.dropna(subset=['isp_mapped'])
               .groupby(['school_id_giga', 'year'])['isp_mapped']
               .agg(lambda s: s.mode().iloc[0]).rename('primary_isp'))
    sch_year = _sy.groupby(['school_id_giga', 'year']).agg(
        dl=('download_speed', 'median'), ul=('upload_speed', 'median'), lat=('latency', 'median'))
    _wkc = (_sy.groupby(['school_id_giga', 'year', 'week'])['download_speed'].quantile(0.95).ge(THR))
    sch_year = sch_year.join(_wkc.groupby(['school_id_giga', 'year']).mean().rename('comp')).join(_pisp_y)

    _pa, _pb = sch_year.xs(YEAR_A, level='year'), sch_year.xs(YEAR_B, level='year')
    pair = _pa.join(_pb, lsuffix='_a', rsuffix='_b', how='inner').dropna(
        subset=['primary_isp_a', 'primary_isp_b'])
    _switched = pair['primary_isp_a'] != pair['primary_isp_b']
    print(f"Schools measurable in both {YEAR_A} and {YEAR_B}: {len(pair)} | "
          f"kept the same primary ISP: {(~_switched).sum()} | switched: {_switched.sum()}")
    if _switched.any():
        print(f"\nProvider switches ({YEAR_A} -> {YEAR_B}):")
        print((pair.loc[_switched, 'primary_isp_a'] + ' -> ' + pair.loc[_switched, 'primary_isp_b'])
              .value_counts().to_string())
    pair_same = pair[~_switched].copy()

    MIN_PAIRS = 10
    _rows = []
    for _isp, _g in pair_same.groupby('primary_isp_a'):
        if len(_g) < MIN_PAIRS:
            continue
        for _tag, _src, _lbl in _Q3B_METRICS:
            _x, _y = _g[f'{_tag}_a'], _g[f'{_tag}_b']
            _mk = _x.notna() & _y.notna()
            _x, _y = _x[_mk], _y[_mk]
            if len(_x) < MIN_PAIRS:
                continue
            _sc = 100 if _tag == 'comp' else 1
            try:
                _p = _st.wilcoxon(_x, _y).pvalue
            except ValueError:
                _p = float('nan')
            _rows.append({'isp': _isp, 'metric': _lbl, 'n_pairs': len(_x),
                          f'{YEAR_A}': _sc * _x.median(), f'{YEAR_B}': _sc * _y.median(),
                          'change': _sc * (_y - _x).median(), 'p_raw': _p})
    isp_change = pd.DataFrame(_rows)
    if len(isp_change):
        _order = isp_change['p_raw'].sort_values().index
        _run, _m = 0.0, len(isp_change)
        for _rank, _idx in enumerate(_order):
            _run = min(1.0, max(_run, (_m - _rank) * isp_change.loc[_idx, 'p_raw']))
            isp_change.loc[_idx, 'p_holm'] = _run
        isp_change['significant'] = isp_change['p_holm'] < 0.05
        print(f"\nPer-ISP change {YEAR_A} -> {YEAR_B} (same-ISP paired schools, "
              f"Holm-corrected Wilcoxon):")
        display(isp_change.set_index(['isp', 'metric']).sort_index().round(3))
        _sig = isp_change[isp_change['significant']]
        print("VERDICT: " + ("; ".join(
            f"{r['isp']} — {r['metric']} changed by {r['change']:+.1f} (Holm p={r['p_holm']:.4f})"
            for _, r in _sig.iterrows()) if len(_sig) else
            f"no ISP shows a statistically significant {YEAR_A} -> {YEAR_B} change "
            f"(all Holm-adjusted p >= 0.05)."))
    else:
        print(f"\nNo ISP has >= {MIN_PAIRS} same-ISP paired schools — no test run.")

In [ ]:
# =============================================================================
# Q3c — ARE THE DIFFERENCES BETWEEN ISPs SIGNIFICANT? (2025, 2026, both)
# =============================================================================
# Cross-sectional, school-level: each school's median download/upload/latency over the window
# (analysable months, school-hours weekday), grouped by its primary ISP IN THAT
# WINDOW. Kruskal-Wallis for the overall question; pairwise Mann-Whitney U with
# Holm correction for which pairs differ. School = unit of analysis.
# Self-contained: no shared temporaries from other cells.
from itertools import combinations as _comb
MIN_SCHOOLS_ISP = 10

def _school_isp_window(year_prefixes):
    _w = mv_school[mv_school['month_qualified']
                   & mv_school['month'].str[:4].isin(year_prefixes)]
    _sc = _w.groupby('school_id_giga').agg(
        dl_p50=('download_speed', 'median'),
        ul_p50=('upload_speed', 'median'),
        latency_p50=('latency', 'median')
    )
    _ip = (_w.dropna(subset=['isp_mapped']).groupby('school_id_giga')['isp_mapped']
           .agg(lambda s: s.mode().iloc[0]).rename('primary_isp'))
    return _sc.join(_ip).dropna()

def _holm(pvals):
    _order = sorted(range(len(pvals)), key=lambda k: pvals[k])
    _adj, _run, _m = [0.0] * len(pvals), 0.0, len(pvals)
    for _rank, _k in enumerate(_order):
        _run = min(1.0, max(_run, (_m - _rank) * pvals[_k]))
        _adj[_k] = _run
    return _adj

def _cross_isp(label, year_prefixes):
    _sch = _school_isp_window(year_prefixes)
    _cnt = _sch.groupby('primary_isp').size()
    _isps = _cnt[_cnt >= MIN_SCHOOLS_ISP].index.tolist()
    if len(_isps) < 2:
        print(f"\n=== {label}: too few ISPs with >= {MIN_SCHOOLS_ISP} schools ==="); return None
    metrics = {
        'dl_p50': ("Download", "Mbps", False),
        'ul_p50': ("Upload", "Mbps", False),
        'latency_p50': ("Latency", "ms", True)
    }
    results = {}
    for metric, (label_print, units, is_lower_better) in metrics.items():
        _grp = {g: _sch.loc[_sch['primary_isp'] == g, metric] for g in _isps}
        _kw = _st.kruskal(*_grp.values())
        print(f"\n{'=' * 74}\n{label} — school-level median {label_print.lower()} by primary ISP "
              f"({len(_isps)} ISPs, {sum(len(v) for v in _grp.values())} schools)\n{'=' * 74}")
        sort_key = (lambda g: _grp[g].median()) if not is_lower_better else (lambda g: _grp[g].median())
        for _g in sorted(_isps, key=sort_key, reverse=not is_lower_better):
            med = _grp[_g].median()
            print(f"  {_g:16s} n={len(_grp[_g]):>3}  median {med:6.1f} {units}")
        print(f"  Kruskal-Wallis: H = {_kw.statistic:.1f}, p = {_kw.pvalue:.2e}"
              f"{'  -> ISPs differ significantly' if _kw.pvalue < 0.05 else '  -> no overall difference'}")
        _rows = [{
            'pair': f"{a} vs {b}",
            'median_1': _grp[a].median(),
            'median_2': _grp[b].median(),
            'p_raw': _st.mannwhitneyu(_grp[a], _grp[b], alternative='two-sided').pvalue
        } for a, b in _comb(sorted(_isps), 2)]
        _pw = pd.DataFrame(_rows)
        _pw['p_holm'] = _holm(_pw['p_raw'].tolist())
        _pw['significant'] = _pw['p_holm'] < 0.05
        print(f"\nPairwise Mann-Whitney U ({label_print}):")
        display(_pw.sort_values('p_holm').reset_index(drop=True).round(4))
        results[metric] = _pw
    return _sch

_by_year = {}
for _y in YEARS_USED:
    _by_year[_y] = _cross_isp(f'{_y} only', [_y])
sch_both = _cross_isp(' + '.join(YEARS_USED) + ' combined', YEARS_USED) if len(YEARS_USED) > 1 else None
sch_2025 = _by_year.get(BASELINE_YEAR)
sch_2026 = _by_year.get(COMPARE_YEAR) if COMPARE_YEAR else None

# ── Within each ISP: did its schools' distribution shift between years? ───────
# Unpaired (compares the ISP's school population each year, so schools that
# joined or left the provider are included — complements Q3b's paired test).
print(f"\n{'=' * 74}\nWithin-ISP change {BASELINE_YEAR} -> {COMPARE_YEAR} (unpaired, Mann-Whitney U, Holm-corrected)"
      f"\n{'=' * 74}")

for metric, label_print, units, is_lower_better in [
    ('dl_p50', 'Download', 'Mbps', False),
    ('ul_p50', 'Upload', 'Mbps', False),
    ('latency_p50', 'Latency', 'ms', True)
]:
    _rows = []
    for _isp in (sorted(set(sch_2025['primary_isp']) & set(sch_2026['primary_isp']))
             if (sch_2025 is not None and sch_2026 is not None) else []):
        _a = sch_2025.loc[sch_2025['primary_isp'] == _isp, metric]
        _b = sch_2026.loc[sch_2026['primary_isp'] == _isp, metric]
        if min(len(_a), len(_b)) < MIN_SCHOOLS_ISP:
            continue
        _rows.append({'isp': _isp,
                      'n_a': len(_a),
                      'n_b': len(_b),
                      'median_a': _a.median(),
                      'median_b': _b.median(),
                      'change': _b.median() - _a.median(),
                      'p_raw': _st.mannwhitneyu(_a, _b, alternative='two-sided').pvalue})
    _yr = pd.DataFrame(_rows)
    if len(_yr):
        _yr['p_holm'] = _holm(_yr['p_raw'].tolist())
        _yr['significant'] = _yr['p_holm'] < 0.05
        print(f"\n{label_print.upper()} — Within-ISP change {BASELINE_YEAR} -> {COMPARE_YEAR} ({units}):")
        display(_yr.set_index('isp').sort_values('change', ascending=(is_lower_better)).round(4))
        _s = _yr[_yr['significant']]
        verdicts = []
        for _, r in _s.iterrows():
            if metric == 'latency_p50':
                change_desc = "improved" if r['change'] < 0 else "declined"
            else:
                change_desc = "improved" if r['change'] > 0 else "declined"
            verdicts.append(
                f"{r['isp']} {change_desc} "
                f"({r['median_a']:.1f} -> {r['median_b']:.1f} {units}, "
                f"Holm p={r['p_holm']:.4f})"
            )
        print("VERDICT: " + ("; ".join(verdicts)
                             if len(verdicts) else
                             f"no ISP's school-level {label_print.lower()} distribution changed significantly between 2025 and 2026."))
print("\nNOTE: the latest year may be partial. The unpaired test above includes schools that"
      "\njoined or left each provider; Q3b's paired test holds the school set fixed.")
 

---
## Annex — data quality

In [ ]:
# =============================================================================
# FAIL REASONS DEEP-DIVE — do invalid tests concentrate in schools or ISPs?
# =============================================================================
# pass_fail_overall is a DATA-VALIDITY flag (NDT7 test reliability), not a
# quality verdict. If failures cluster in specific schools/ISPs, those places
# are systematically under-measured wherever failed tests get excluded.
_dq = m.copy()
_dq['pf'] = _dq['pass_fail_overall'].astype('string').str.lower()
_known = _dq[_dq['pf'].isin(['pass', 'fail'])].copy()
_known['is_fail'] = _known['pf'] == 'fail'
_overall_fr = _known['is_fail'].mean()
print(f"Validity flag: pass {(_dq['pf'] == 'pass').sum():,} | fail {(_dq['pf'] == 'fail').sum():,} "
      f"| not computed {(~_dq['pf'].isin(['pass', 'fail'])).sum():,}")
print(f"Overall fail rate (where computed): {100 * _overall_fr:.1f}%")

# 1. Reasons (a failed test can carry several)
_reasons = (_known.loc[_known['is_fail'], 'reasons_failed_overall'].dropna()
            .str.split(', ').explode().str.strip())
print("\nFail reasons:")
for _r, _n in _reasons.value_counts().head(8).items():
    print(f"  {_r[:58]:58s} {_n:>7,}")

# 2. Schools — volume concentration + highest rates
_sch = (_known.groupby(['school_id_giga', 'school_name'])
        .agg(n=('is_fail', 'size'), fails=('is_fail', 'sum'),
             primary_isp=('isp_mapped', lambda s: s.mode().iloc[0] if s.notna().any() else '?'))
        .reset_index())
_sch['fail_perc'] = 100 * _sch['fails'] / _sch['n']
_top10_share = _sch.nlargest(10, 'fails')['fails'].sum() / max(_sch['fails'].sum(), 1)
print(f"\nSchool concentration: the top 10 schools by fail volume hold {100 * _top10_share:.0f}% "
      f"of all {_sch['fails'].sum():,} fails ({(_sch['fails'] > 0).sum()} schools have any)")
print("Highest fail-RATE schools (n >= 50), with their primary ISP:")
display(_sch[_sch['n'] >= 50].sort_values('fail_perc', ascending=False)
        .head(10)[['school_name', 'primary_isp', 'n', 'fails', 'fail_perc']].round({'fail_perc': 1}))

# 3. ISPs — fail rate vs the overall rate
_isp = (_known.dropna(subset=['isp_mapped']).groupby('isp_mapped')
        .agg(n=('is_fail', 'size'), schools=('school_id_giga', 'nunique'), fails=('is_fail', 'sum'))
        .reset_index())
_isp['fail_perc'] = 100 * _isp['fails'] / _isp['n']
_isp['vs_overall_pp'] = _isp['fail_perc'] - 100 * _overall_fr
print(f"\nFail rate by ISP (n >= 500), overall = {100 * _overall_fr:.1f}%:")
display(_isp[_isp['n'] >= 500].sort_values('fail_perc', ascending=False)
        .round({'fail_perc': 1, 'vs_overall_pp': 1}))

# 4. Triangulation — school effect vs ISP effect
_hi = _sch[(_sch['n'] >= 50) & (_sch['fail_perc'] >= 200 * _overall_fr)]
print(f"\nTriangulation: {len(_hi)} schools fail at >= 2x the overall rate (n >= 50).")
print("Their primary-ISP mix vs all schools:")
_mix = _hi['primary_isp'].value_counts(normalize=True).round(2)
_all_mix = _sch['primary_isp'].value_counts(normalize=True).round(2)
for _i in _mix.index[:6]:
    print(f"  {str(_i)[:24]:24s} {100 * _mix[_i]:4.0f}% of high-fail schools vs {100 * _all_mix.get(_i, 0):4.0f}% of all schools")
print("-> If the mixes match, failures are school-local (device/setup); a large"
      "\n   over-representation points at the ISP's network instead.")

# 5. Fail reasons by ISP — does the failure MODE differ by provider?
_fr = _known[_known['is_fail']].dropna(subset=['isp_mapped']).copy()
_fr = _fr.assign(reason=_fr['reasons_failed_overall'].str.split(', ')).explode('reason')
_fr['reason'] = _fr['reason'].str.strip()
_big = set(_isp.loc[_isp['n'] >= 500, 'isp_mapped'])
_ct = (pd.crosstab(_fr['isp_mapped'], _fr['reason'], normalize='index') * 100)
_ct = _ct.loc[_ct.index.isin(_big)]
_ct = _ct[_ct.mean().sort_values(ascending=False).index].round(0).astype(int)
print("\nFail-reason mix by ISP (% of the ISP's reason mentions):")
display(_ct)

In [ ]:
# =============================================================================
# FOREIGN / ROAMING ISP FLAGS — measurements egressing via out-of-country ISPs
# =============================================================================
# Roaming SIM hotspots tunnel traffic to the SIM's home carrier, so the detected
# ISP is the home network — e.g. Uzbek carriers (Uzbektelekom/UNITEL/COSCOM) at
# two Fiji schools via an Uzbek roaming hotspot (found Aug 2026). Heuristic:
# ISPs serving <= RARE_ISP_MAX_SCHOOLS schools nationally AND a minority share
# at the school. Review the list — rare legitimate local ISPs can appear too.
RARE_ISP_MAX_SCHOOLS = 3

_isp_schools = m.groupby('isp_mapped')['school_id_giga'].nunique()
_rare = set(_isp_schools[_isp_schools <= RARE_ISP_MAX_SCHOOLS].index)
_school_share = (m.groupby(['school_id_giga', 'isp_mapped']).size()
                 / m.groupby('school_id_giga').size()).rename('share')
flag = (m.groupby(['school_id_giga', 'school_name', 'isp_mapped'])
          .agg(n=('isp_mapped', 'size'), lat_med=('latency', 'median'),
               first=('date', 'min'), last=('date', 'max')).reset_index()
          .merge(_school_share.reset_index(), on=['school_id_giga', 'isp_mapped']))
flag = flag[flag['isp_mapped'].isin(_rare) & (flag['share'] < 0.5)]
flag['school_lat_med'] = flag['school_id_giga'].map(m.groupby('school_id_giga')['latency'].median())
flag = flag.sort_values('n', ascending=False)

if len(flag):
    print(f"⚠️ Possible roaming/foreign or ad-hoc ISPs: {len(flag)} school×ISP combos, "
          f"{int(flag['n'].sum()):,} measurements — verify before trusting these schools' baselines")
    display(flag[['school_name', 'isp_mapped', 'n', 'share', 'lat_med', 'school_lat_med', 'first', 'last']]
            .round({'share': 2, 'lat_med': 0, 'school_lat_med': 0}).head(20))
else:
    print("✓ No rare-ISP anomalies flagged")

---
## Exports

In [ ]:
# =============================================================================
# SUPERSET-READY EXPORTS
# =============================================================================
outd = Path(CACHE_DIR) / 'superset'
outd.mkdir(exist_ok=True)

_daily = (mv.assign(_d=mv['date'].dt.date).groupby('_d')
          .agg(n_tests=('school_id_giga', 'size'), schools=('school_id_giga', 'nunique')))
_daily['scope_schools'] = (mv[mv['in_scope']].assign(_d=lambda x: x['date'].dt.date)
                         .groupby('_d')['school_id_giga'].nunique())
_daily.fillna(0).astype(int).reset_index().to_csv(outd / f'{COUNTRY_ISO3.lower()}_reporting_daily.csv', index=False)
cov.reset_index().to_csv(outd / f'{COUNTRY_ISO3.lower()}_reporting_monthly.csv', index=False)
base.reset_index().to_csv(outd / f'{COUNTRY_ISO3.lower()}_school_baseline_secondary.csv', index=False)
gap.to_csv(outd / f'{COUNTRY_ISO3.lower()}_secondary_coverage_gap.csv', index=False)
sw.to_csv(outd / f'{COUNTRY_ISO3.lower()}_school_week_isp.csv', index=False)
(mv_school[mv_school['month'].str[:4].isin(YEARS_USED) & mv_school['month_qualified']]
 .dropna(subset=['isp_mapped'])
 .groupby(['isp_mapped', 'month'])
 .agg(n_tests=('download_speed', 'size'), schools=('school_id_giga', 'nunique'),
      dl_p50=('download_speed', 'median'), ul_p50=('upload_speed', 'median'),
      lat_p50=('latency', 'median'),
      pct_tests_below_20=('download_speed', lambda s: 100 * (s < 20).mean()))
 .reset_index().to_csv(outd / f'{COUNTRY_ISO3.lower()}_isp_summary_monthly.csv', index=False))
isp_rank.reset_index().to_csv(outd / f'{COUNTRY_ISO3.lower()}_isp_threshold_compliance.csv', index=False)

for _f in sorted(outd.glob('*.csv')):
    print(f"{_f.name:40s} {sum(1 for _ in open(_f)) - 1:>8,} rows")